In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("votacoes_com_llm.csv")
df.tail(5)

,id,data,idOrgao,siglaOrgao,aprovacao,votosSim,votosNao,votosOutros,descricao,ano,...,tipoAutor,idDeputadoAutor,nomeAutor,data_ref,legislatura,author_prev_community,prev_community_0_size,prev_community_1_size,prev_community_2_size,classified_llm
78813,347421-202,2024-12-19,180,PLEN,NaN,4,360,1,Suprimido o texto. Sim: 4; Não: 360; Total: 364.,2024,...,Deputado(a),73778.0,Luiz Carlos Hauly,1889,57,NaN,441.0,155.0,0,NaN
78814,347421-202,2024-12-19,180,PLEN,NaN,4,360,1,Suprimido o texto. Sim: 4; Não: 360; Total: 364.,2024,...,Deputado(a),74283.0,Vicentinho,1889,57,1.0,441.0,155.0,0,NaN
78815,347421-202,2024-12-19,180,PLEN,NaN,4,360,1,Suprimido o texto. Sim: 4; Não: 360; Total: 364.,2024,...,Deputado(a),141531.0,Rodrigo de Castro,1889,57,0.0,441.0,155.0,0,NaN
78816,347421-205,2024-12-19,180,PLEN,NaN,349,143,1,Mantido o texto. Sim: 349; Não: 143; Abstenção...,2024,...,Deputado(a),74063.0,Fernando de Fabinho,1889,57,NaN,441.0,155.0,0,NaN
78817,2473375-80,2024-12-19,180,PLEN,1.0,0,0,0,Aprovada a Redação Final assinada pelo relator...,2024,...,Deputado(a),160674.0,Hugo Motta,1889,57,0.0,441.0,155.0,0,Approval of Final Wording


In [4]:
df.loc[df['aprovacao'].isna(), 'aprovacao'] = (df['votosSim'] > df['votosNao']).astype(float)
df.tail(5)

,id,data,idOrgao,siglaOrgao,aprovacao,votosSim,votosNao,votosOutros,descricao,ano,...,tipoAutor,idDeputadoAutor,nomeAutor,data_ref,legislatura,author_prev_community,prev_community_0_size,prev_community_1_size,prev_community_2_size,classified_llm
78813,347421-202,2024-12-19,180,PLEN,0.0,4,360,1,Suprimido o texto. Sim: 4; Não: 360; Total: 364.,2024,...,Deputado(a),73778.0,Luiz Carlos Hauly,1889,57,NaN,441.0,155.0,0,NaN
78814,347421-202,2024-12-19,180,PLEN,0.0,4,360,1,Suprimido o texto. Sim: 4; Não: 360; Total: 364.,2024,...,Deputado(a),74283.0,Vicentinho,1889,57,1.0,441.0,155.0,0,NaN
78815,347421-202,2024-12-19,180,PLEN,0.0,4,360,1,Suprimido o texto. Sim: 4; Não: 360; Total: 364.,2024,...,Deputado(a),141531.0,Rodrigo de Castro,1889,57,0.0,441.0,155.0,0,NaN
78816,347421-205,2024-12-19,180,PLEN,1.0,349,143,1,Mantido o texto. Sim: 349; Não: 143; Abstenção...,2024,...,Deputado(a),74063.0,Fernando de Fabinho,1889,57,NaN,441.0,155.0,0,NaN
78817,2473375-80,2024-12-19,180,PLEN,1.0,0,0,0,Aprovada a Redação Final assinada pelo relator...,2024,...,Deputado(a),160674.0,Hugo Motta,1889,57,0.0,441.0,155.0,0,Approval of Final Wording


In [5]:
# Create authors_pop column to calculate author popularity by theme and classification
import numpy as np

# Convert date column to datetime
df['data'] = pd.to_datetime(df['data'])
df = df.sort_values('data')

# Initialize the authors_pop column with NaN
df['authors_pop'] = np.nan

# Iterate through each row in the dataframe
for idx, row in df.iterrows():
    author_id = row['idDeputadoAutor']
    current_date = row['data']
    current_tema = row['tema']
    current_classification = row['classified_llm']
    
    # Skip if author_id is NaN
    if pd.isna(author_id):
        continue
    
    # Find all previous propositions by this author with the same tema and classification
    mask = (
        (df['idDeputadoAutor'] == author_id) & 
        (df['data'] < current_date) & 
        (df['tema'] == current_tema) & 
        (df['classified_llm'] == current_classification)
    )
    
    previous_props = df.loc[mask]
    
    # Calculate popularity only if there are previous propositions
    if len(previous_props) > 0:
        popularity = previous_props['aprovacao'].mean()
        df.at[idx, 'authors_pop'] = popularity
        
# Show the dataframe with the new column
df[['idDeputadoAutor', 'nomeAutor', 'data', 'tema', 'classified_llm', 'aprovacao', 'authors_pop']].head(10)


,idDeputadoAutor,nomeAutor,data,tema,classified_llm,aprovacao,authors_pop
0,74218.0,Walter Pinheiro,2003-02-19,Administração Pública,NaN,1.0,NaN
1,74218.0,Walter Pinheiro,2003-02-19,Administração Pública,NaN,1.0,NaN
2,73947.0,Maurício Rabelo,2003-03-18,Direitos Humanos e Minorias,Approval of Requests,1.0,NaN
3,73947.0,Maurício Rabelo,2003-03-18,Direitos Humanos e Minorias,Approval of Requests,1.0,NaN
4,74082.0,Paulo Rocha,2003-03-20,Trabalho e Emprego,Approval of Final Wording,1.0,NaN
5,74082.0,Paulo Rocha,2003-03-20,Trabalho e Emprego,Approval of Final Wording,1.0,NaN
6,73558.0,Robson Tuma,2003-03-20,Economia,Approval of Requests,1.0,NaN
7,73558.0,Robson Tuma,2003-03-20,Economia,Approval of Requests,1.0,NaN
17,73431.0,Antonio Carlos Pannunzio,2003-04-02,Comunicações,Approval of Requests,1.0,NaN
16,74416.0,Eduardo Campos,2003-04-02,Administração Pública,Approval of Requests,1.0,NaN


In [6]:
df.tail(5)

,id,data,idOrgao,siglaOrgao,aprovacao,votosSim,votosNao,votosOutros,descricao,ano,...,idDeputadoAutor,nomeAutor,data_ref,legislatura,author_prev_community,prev_community_0_size,prev_community_1_size,prev_community_2_size,classified_llm,authors_pop
77493,347421-224,2024-12-19,180,PLEN,1.0,358,129,1,Mantido o texto. Sim: 358; Não: 129; Abstenção...,2024,...,74345.0,Evandro Milhomen,1889,57,NaN,441.0,155.0,0,NaN,NaN
77492,347421-224,2024-12-19,180,PLEN,1.0,358,129,1,Mantido o texto. Sim: 358; Não: 129; Abstenção...,2024,...,141451.0,Ilderlei Cordeiro,1889,57,NaN,441.0,155.0,0,NaN,NaN
77491,347421-224,2024-12-19,180,PLEN,1.0,358,129,1,Mantido o texto. Sim: 358; Não: 129; Abstenção...,2024,...,141539.0,Sebastião Bala Rocha,1889,57,NaN,441.0,155.0,0,NaN,NaN
77489,347421-224,2024-12-19,180,PLEN,1.0,358,129,1,Mantido o texto. Sim: 358; Não: 129; Abstenção...,2024,...,141386.0,Eudes Xavier,1889,57,NaN,441.0,155.0,0,NaN,NaN
78817,2473375-80,2024-12-19,180,PLEN,1.0,0,0,0,Aprovada a Redação Final assinada pelo relator...,2024,...,160674.0,Hugo Motta,1889,57,0.0,441.0,155.0,0,Approval of Final Wording,NaN


In [8]:
unique_authors_pop = df['authors_pop'].unique()
print("Unique values in 'authors_pop':", unique_authors_pop)

authors_pop_counts = df['authors_pop'].value_counts(dropna=False)
print("\nCount of each unique value in 'authors_pop':")
print(authors_pop_counts)



Unique values in 'authors_pop': [       nan 1.         0.75       0.66666667 0.6        0.5
 0.42857143 0.375      0.44444444 0.85714286 0.38461538 0.33333333
 0.18181818 0.14285714 0.13636364 0.2        0.17391304 0.
 0.8        0.69230769 0.26666667 0.48       0.25       0.71428571
 0.57142857 0.83333333 0.625      0.875      0.36842105 0.58333333
 0.4        0.4375     0.36363636 0.34782609 0.30769231 0.35294118
 0.22222222 0.45454545 0.16666667 0.53846154 0.77777778 0.54545455
 0.7        0.88888889 0.81818182 0.55555556 0.38888889 0.42105263
 0.41666667 0.90909091 0.9        0.72727273 0.76923077 0.78571429
 0.9375     0.93333333 0.63636364 0.73333333 0.92307692 0.94117647
 0.92857143 0.91666667 0.61538462 0.64285714 0.76470588 0.8125
 0.70588235 0.78947368 0.53333333 0.94444444 0.82352941 0.73913043
 0.28571429 0.95       0.84615385 0.6875     0.94736842 0.95238095
 0.91304348 0.68421053 0.9047619  0.88235294 0.86363636 0.64705882
 0.58823529 0.84210526 0.61111111 0.85       0.63

In [9]:
# Save the dataframe with the new author popularity column
df.to_csv("votacoes_com_llm_pop.csv", index=False)
print("Dataframe saved with new authors_pop column")


Dataframe saved with new authors_pop column
